# EDA: `fraud-review-queue`, timeboxed to a single day

> **The rules of the timebox.**
> 1. The 5 questions below were written **before** the notebook was opened. No
>    more get added along the way.
> 2. Each question exists because it **feeds a concrete decision downstream**.
>    A figure that answers none of the 5 does not go in.
> 3. **Eight figures at most. One day. Answer and close.**
> 4. The V-columns are **not** studied one by one today (design.md §5.4). If
>    they show up, it is only in Q4, through their null pattern.
>
> Today's risk is not difficulty, it is **seduction**. There are interesting
> patterns to find in the V-columns and they are tempting to chase. Do not.

**The answers are still to be written.** What is here is the questions, the
*why*, and the starting code. The interpretation is the deliverable.


## Setup

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)

# Adjust if ingest.py was run against another path.
TXN = "data/processed/transactions.parquet"
IDY = "data/processed/identity.parquet"

con = duckdb.connect()

# The split boundaries (design.md §4), to mark them on the figures.
TRAIN_END, EMBARGO_END, CALIB_END = 119, 129, 155

con.execute(f"SELECT COUNT(*) AS n, AVG(isFraud) AS fraud_rate FROM read_parquet('{TXN}')").df()


## Q1: where does fraud live along the **amount** axis?

**Why this is THE question.** The magnitude of the whole thesis depends on it.
`V` peaks at `p` around 0.25 by algebra, which is guaranteed, but whether the
**difference in dollars** between ranking by value and ranking by score is
*large* depends on there being **high-amount** fraud. If the IEEE-CIS frauds
are nearly all small, the effect can be trivial.

**The decision it feeds:** it sets a realistic expectation **before** the
single evaluation on test, and looking at it today removes the temptation to
bend parameters on the critical day.


In [ ]:
# TransactionAmt by class (log-x), and fraud rate by amount decile.
df = con.execute(f"""
    SELECT TransactionAmt, isFraud
    FROM read_parquet('{TXN}')
""").df()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bins = np.logspace(0, np.log10(df.TransactionAmt.max()), 60)
for label, flag in (("legit", 0), ("fraud", 1)):
    ax[0].hist(df.loc[df.isFraud == flag, "TransactionAmt"],
               bins=bins, alpha=.5, density=True, label=label)
ax[0].set_xscale("log")
ax[0].set_xlabel("TransactionAmt")
ax[0].set_ylabel("density")
ax[0].legend()

df["amt_decile"] = pd.qcut(df.TransactionAmt, 10, labels=False, duplicates="drop")
rate = df.groupby("amt_decile").isFraud.mean()
ax[1].bar(rate.index, rate.values)
ax[1].set_xlabel("amount decile")
ax[1].set_ylabel("fraud rate")
plt.tight_layout()

# TODO: does fraud concentrate on small amounts, or is there mass at the high
# end? What fraction of the fraudulent *money* sits in the top amount quartile?


**Answer.** Fraud is spread almost evenly along the amount axis, and the fraudulent
**money** is not.

The fraud rate by amount decile runs 5.6 %, 3.2 %, 3.2 %, 1.9 %, 2.9 %, 3.6 %, 2.0 %,
4.3 %, 3.8 %, 5.1 %. The top decile and the bottom decile are within 10 % of each other,
so **amount barely predicts fraud at all**: the median fraudulent transaction is $75.00
against $68.50 for a legitimate one.

The dollars tell a completely different story. The top amount quartile starts at $125.00
and holds **31.5 % of the fraudulent cases but 75.5 % of the fraudulent money**. Approving
everything would lose $3,083,845 over these 182 days, and three quarters of that sits in
one quartile of transactions.

**What it feeds.** This is the empirical precondition of the whole thesis, and it holds.
If `p` barely moves with the amount while the loss moves by a factor of three, then
ranking a queue by `p` sorts on the axis that carries no money. `V = min(cost_approve,
cost_block) - r` grows with the amount, so it sorts on the axis that does. The effect
being measured later is not an artefact of the cost parameters: it is there in the raw
amounts, before any model.

## Q2: is the **temporal split** viable? Are there enough positives in calibration?

**Why.** The split (train 0-119, embargo 120-129, calib 130-155, test 156 on)
only works if every partition has volume and, critically, if the **calibration
partition has enough positives** to fit an isotonic regression (design.md
§6.3). If the fraud rate collapses over the last days, the calibrator and the
test are left without signal.

**The decision it feeds:** it confirms, or corrects, the boundary days of
`SplitConfig`. If calib holds few positives, Platt over isotonic.


In [ ]:
daily = con.execute(f"""
    SELECT day, COUNT(*) AS n, SUM(isFraud) AS n_fraud, AVG(isFraud) AS rate
    FROM read_parquet('{TXN}')
    GROUP BY day ORDER BY day
""").df()

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax[0].plot(daily.day, daily.n)
ax[0].set_ylabel("transactions per day")
ax[1].plot(daily.day, daily.rate)
ax[1].set_ylabel("fraud rate per day")
ax[1].set_xlabel("day")
for a in ax:
    for d in (TRAIN_END, EMBARGO_END, CALIB_END):
        a.axvline(d, ls="--", c="k", alpha=.4)
plt.tight_layout()

# Positives per partition: the number that decides Platt against isotonic.
con.execute(f"""
    SELECT
      CASE WHEN day <= {TRAIN_END} THEN '1_train'
           WHEN day <= {EMBARGO_END} THEN '2_embargo'
           WHEN day <= {CALIB_END} THEN '3_calib'
           ELSE '4_test' END AS part,
      COUNT(*) AS n, SUM(isFraud) AS n_fraud, AVG(isFraud) AS rate
    FROM read_parquet('{TXN}')
    GROUP BY part ORDER BY part
""").df()


**Answer.** Viable, with room to spare, and no boundary needs moving.

| Partition | Days | Rows | Fraud | Rate |
|---|---|---|---|---|
| train | 1 to 119 | 410,601 | 14,419 | 3.51 % |
| embargo | 120 to 129 | 31,765 | 1,116 | 3.51 % |
| calib | 130 to 155 | 72,984 | 2,473 | 3.39 % |
| test | 156 to 182 | 75,190 | 2,655 | 3.53 % |

**Calibration holds 2,473 positives**, which is the number this question exists to check.
Isotonic regression is a step function fitted on the positives, and a few hundred would
have made it jagged; a few thousand is comfortable. Both calibrators stay on the table,
and the choice is left to the Brier score on a temporal holdout inside calib rather than
being forced here.

The fraud rate is flat across all four partitions, between 3.39 % and 3.53 %, so no
partition is being asked to learn or verify a different problem from the others. Daily
volume runs from 2,048 to 6,852 transactions across the whole dataset, so a capacity of
1 % never rounds down to zero and the queue is always exercised. On the test partition it
comes to 20 to 32 reviews a day.

**What it feeds.** `SplitConfig` is confirmed as written. The one thing to carry forward
is that the daily fraud rate swings between 1.1 % and 7.0 % day to day, which is worth
remembering when a weekly drift figure moves by a few points.

## Q3: does `D1n = day - D1` behave as a **per-card constant**? (is the UID viable?)

**Why.** The UID is built as `card1 + addr1 + D1n`, with the backward-looking
features on top of it. All of that assumes `D1n`, an approximation of the
card's registration date, is stable within a single card. If it is not, the
entity resolution does not work and needs rethinking.

**The decision it feeds:** it de-risks the feature engineering **now**, with a
cheap check, instead of discovering the problem halfway through it.


In [ ]:
probe = con.execute(f"""
    SELECT
        card1, addr1,
        COUNT(*) AS n,
        COUNT(DISTINCT (day - D1)) AS n_distinct_d1n
    FROM read_parquet('{TXN}')
    WHERE D1 IS NOT NULL AND addr1 IS NOT NULL
    GROUP BY card1, addr1
    HAVING COUNT(*) >= 3
""").df()

# Were D1n perfectly constant per (card1, addr1), n_distinct_d1n == 1.
share_constant = (probe.n_distinct_d1n == 1).mean()
print(f"(card1, addr1) groups with >=3 txns: {len(probe):,}")
print(f"Share with a unique D1n: {share_constant:.1%}")

# TODO: is that high enough to trust the UID? What happens to the tail of
# groups with several D1n values: noise in addr1, or does the proxy break?


**Answer.** No, and the question turns out to be pointed at the wrong assumption. `D1n`
is **not** constant within `(card1, addr1)`: of the 17,177 groups with at least three
transactions, only **15.7 % have a single `D1n`**, and those cover **3.2 % of the
transactions**.

The tempting reading is that `D1 = day - D1n` is noisy. It is not. Widening the tolerance
from an exact match to a 30-day window moves the share of groups from 15.7 % to only
21.1 %, and among the non-constant groups the **median spread of `D1n` is 208 days**, with
a 90th percentile of 615. Those are not rounding artefacts. They are different cards.

The real finding is about the key, not the proxy: `card1` takes 13,553 distinct values and
`addr1` only 332, so `(card1, addr1)` is closer to an issuer-and-region bucket than to a
customer. `D1n` is precisely what splits that bucket into something customer-shaped, which
is why the UID is built from all three (`design.md` §5.2).

**What it feeds.** The UID is kept, with its limitation stated rather than assumed away.
It produces 199,070 UIDs over the 523,746 transactions that have one, and **58 % of those
UIDs carry a single transaction**, with a median of one. The entity resolution therefore
**over-splits**: most rows get little or no history, and the backward-looking features are
sparse.

That is the safe direction to be wrong in, and the distinction matters. A UID that merged
distinct customers would manufacture history across strangers and inflate the features. A
UID that splits one customer into several only withholds history from itself. The cost is
coverage, not correctness, and it is paid in feature strength: the five `uid_` features
end up carrying 2.6 % of the model's total gain.

## Q4: identity coverage and null patterns (today's only look at the V-columns)

**Why.** Two feature-engineering decisions come out of here:

1. `train_identity` covers only a fraction of the transactions, so the join is
   a **left join, not an inner one** (design.md §3.2). The exact number matters.
2. The 339 V-columns group into Vesta blocks that **share a null pattern**
   (§5.4). How many blocks there are decides whether to pick one representative
   per block or hand them all to LightGBM whole.

**The decision it feeds:** the identity join strategy and the V-column
strategy. **No column-by-column archaeology.**


In [ ]:
n_txn = con.execute(f"SELECT COUNT(*) FROM read_parquet('{TXN}')").fetchone()[0]
n_idy = con.execute(f"SELECT COUNT(*) FROM read_parquet('{IDY}')").fetchone()[0]
print(f"Identity coverage: {n_idy:,} / {n_txn:,} = {n_idy/n_txn:.1%}  -> left join.")

# Null patterns of the V-columns: how many distinct blocks there are.
vcols = [c for c in con.execute(f"SELECT * FROM read_parquet('{TXN}') LIMIT 0").df().columns
         if c.startswith("V")]
sample = con.execute(
    f"SELECT {', '.join(vcols)} FROM read_parquet('{TXN}') USING SAMPLE 20000 ROWS"
).df()
null_signature = sample.isnull().mean().round(3)          # share of nulls per V-col
n_blocks = null_signature.nunique()
print(f"V-columns: {len(vcols)}  ->  {n_blocks} distinct null patterns (Vesta blocks).")

# TODO: how many blocks? One representative per block, or all of them to LightGBM?


**Answer.** Identity covers **144,233 of 590,540 transactions, 24.4 %**. A left join is
the only correct choice: an inner join would silently discard three quarters of the
dataset, and the absence of identity data is itself a signal that LightGBM consumes
natively as NaN.

The 339 V-columns fall into **15 blocks by exact null pattern** over the full dataset, the
largest holding 46 columns and the smallest 11: 46, 43, 32, 31, 23, 22, 20, 19, 18, 18,
18, 16, 11, 11, 11. Columns in a block are null on precisely the same rows, which is the
signature of having been derived from one source.

A caution about how that number is obtained. Grouping by the null **rate** rounded to
three decimals, on a 20,000-row sample, gives 13 rather than 15: two pairs of blocks share
a null rate without sharing a pattern, and the sample blurs the rest. The exact mask is
cheap and it is what `fraudq.diagnostics` uses.

**What it feeds.** All 339 go to LightGBM, option 3 of `design.md` §5.4, and importance is
reported instead of one column being picked per block. The diagnostics justify that after
the fact: the 339 V-columns account for 39.0 % of total gain, while the 14 C-columns
account for 19.6 %. **Per column that is roughly 35 times more gain in a C than in a V.**
Spending the timebox on V-column archaeology would have been spending it on the least
informative columns in the dataset.

## Q5: which base categoricals separate fraud? (`ProductCD`, `card4`, `card6`, email)

**Why.** Before investing in features, confirm which low-cardinality
categoricals carry signal, and look at the cardinality of the high ones
(`card1`, `addr1`) for the frequency encoding (design.md §5.1). Cheap and
directly actionable.

**The decision it feeds:** the set of base features, and which categoricals
deserve a frequency encoding computed **on train only**.


In [ ]:
for colname in ["ProductCD", "card4", "card6"]:
    print(f"=== {colname} ===")
    print(con.execute(f"""
        SELECT {colname}, COUNT(*) AS n, AVG(isFraud) AS fraud_rate
        FROM read_parquet('{TXN}')
        GROUP BY {colname} ORDER BY n DESC
    """).df().to_string(index=False))
    print()

# Cardinality of the high-cardinality ones, the frequency-encoding candidates.
print("=== cardinality ===")
print(con.execute(f"""
    SELECT
      COUNT(DISTINCT card1) AS card1,
      COUNT(DISTINCT addr1) AS addr1,
      COUNT(DISTINCT P_emaildomain) AS p_email
    FROM read_parquet('{TXN}')
""").df().to_string(index=False))

# TODO: which categories sit well above the base fraud rate? Which email
# domains get grouped into 'other'?


**Answer.** Three of the four separate fraud clearly, and the split is large enough to be
worth encoding. The base rate is 3.50 %.

- **`ProductCD`** is the strongest: `C` runs at **11.7 %**, more than three times base,
  against 2.0 % for `W`, which is 74 % of the volume. A 5.7-fold spread across five
  categories.
- **`card6`**: credit at **6.7 %** against debit at 2.4 %, on 149k and 440k transactions.
  Clean, high volume, nearly a factor of three.
- **`card4`**: essentially flat. Visa 3.5 %, Mastercard 3.4 %, Amex 2.9 %. Discover shows
  7.7 % but on only 6,651 transactions. This one carries little.
- **Email domain** does separate: `outlook.com` at 9.5 % and `hotmail.com` at 5.3 %
  against `icloud.com` and `comcast.net` at 3.1 %, with `gmail.com` at 4.4 % on 228k
  transactions.

Cardinalities for the encoding decision: `card1` 13,553 distinct values, `addr1` 332,
`P_emaildomain` 59, `R_emaildomain` 60.

**What it feeds.** `card1` and `addr1` are far too high-cardinality for one-hot and go to
frequency encoding, **fitted on train only** so that the calibration and test distributions
cannot shape the training representation (`design.md` §5.1). The email domains are reduced
to their base provider and frequency encoded on the same terms. `ProductCD`, `card4` and
`card6` are low-cardinality and are left for the model to handle directly.

Confirmed after the fact by the diagnostics: `card1_freq` is the third feature by gain and
`addr1_freq` the twelfth, both ahead of their raw counterparts. The encoding earned its
place.

## Closing the timebox

- [x] The 5 questions are answered in prose, not only in figures.
- [x] At most 8 figures. No figure that answers none of the 5.
- [x] The split boundary days are confirmed (Q2), or the adjustment is noted.
- [x] Decided: Platt or isotonic (Q2), UID yes or no (Q3), V-column strategy (Q4).
- [x] **Closed the notebook. No further exploring of the V-columns.**
